# PsychoCounsel — One-cell quickstart

Run this single cell. It will:
1. Install `llama-cpp-python[server]` (skips if already installed)
2. Mount Google Drive (skips if already mounted)
3. Download `cloudflared` (skips if already on disk)
4. Boot the LLM server, wait until `/docs` responds
5. Open a Cloudflare tunnel and print the public URL

Re-running is safe — every step is idempotent and old processes are killed first.

In [ ]:
import os, sys, time, subprocess, threading, importlib

MODEL_PATH = "/content/drive/MyDrive/PsychoCounsel-Llama3-8B.Q4_K_M.gguf"
PORT = 8000
SERVER_TIMEOUT = 180

def step(msg):
    print(f"\n\u2192 {msg}", flush=True)

# ---------------------------------------------------------------
# 0) Kill any previous server / tunnel
# ---------------------------------------------------------------
step("Cleaning up any previous server / tunnel")
subprocess.run(["pkill", "-f", "llama_cpp.server"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)

# ---------------------------------------------------------------
# 1) Verify GPU + install llama-cpp-python[server]
# ---------------------------------------------------------------
try:
    has_gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise RuntimeError("No GPU. In Colab: Runtime \u2192 Change runtime type \u2192 T4 GPU")
step("GPU detected \u2705")

def _try_import():
    try:
        if "llama_cpp" in sys.modules: del sys.modules["llama_cpp"]
        importlib.import_module("llama_cpp")
        return True
    except (ImportError, OSError, RuntimeError):
        return False

if _try_import():
    step("llama_cpp already installed \u2705")
else:
    step("Installing llama-cpp-python[server] (~1.5 GB)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python[server]",
        "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu122"], check=True)
    if not _try_import():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
            "llama-cpp-python[server]",
            "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu122"], check=True)
        if not _try_import():
            raise RuntimeError("Could not load llama_cpp \u2014 check !nvidia-smi")

# ---------------------------------------------------------------
# 2) Mount Drive
# ---------------------------------------------------------------
if not os.path.ismount("/content/drive"):
    step("Mounting Google Drive")
    from google.colab import drive
    drive.mount("/content/drive")
else:
    step("Drive already mounted \u2705")
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found at {MODEL_PATH}")

# ---------------------------------------------------------------
# 3) Download cloudflared if missing
# ---------------------------------------------------------------
if not os.path.exists("./cloudflared"):
    step("Downloading cloudflared")
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "cloudflared"], check=True)
    os.chmod("./cloudflared", 0o755)
else:
    step("cloudflared ready \u2705")

# ---------------------------------------------------------------
# 4) Boot LLM server (logs suppressed — only errors shown)
# ---------------------------------------------------------------
step("Starting LLM server...")
server = subprocess.Popen(
    [sys.executable, "-m", "llama_cpp.server",
     "--model", MODEL_PATH, "--host", "0.0.0.0", "--port", str(PORT),
     "--n_gpu_layers", "-1", "--n_ctx", "2048", "--chat_format", "chatml"],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True,
)

# Stream stderr in background but only keep last 5 lines for error diagnosis
_server_tail = []
def _capture_err():
    for line in server.stderr:
        _server_tail.append(line.rstrip())
        if len(_server_tail) > 5: _server_tail.pop(0)
threading.Thread(target=_capture_err, daemon=True).start()

import urllib.request, urllib.error
deadline = time.time() + SERVER_TIMEOUT
ready = False
dots = 0
while time.time() < deadline:
    if server.poll() is not None:
        print("\n".join(_server_tail))
        raise RuntimeError("Server exited \u2014 see logs above")
    try:
        with urllib.request.urlopen(f"http://localhost:{PORT}/v1/models", timeout=2) as r:
            if r.status == 200:
                ready = True
                break
    except (urllib.error.URLError, ConnectionResetError, TimeoutError):
        pass
    dots += 1
    if dots % 5 == 0: print(".", end="", flush=True)
    time.sleep(2)

if not ready:
    print("\n".join(_server_tail))
    raise TimeoutError("Server did not start in time")
step("LLM server ready \u2705")

# ---------------------------------------------------------------
# 5) Cloudflare tunnel (only URL printed)
# ---------------------------------------------------------------
step("Opening tunnel...")
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

public_url = None
t_deadline = time.time() + 60
while time.time() < t_deadline:
    line = tunnel.stdout.readline()
    if not line:
        if tunnel.poll() is not None:
            raise RuntimeError("cloudflared exited before printing URL")
        continue
    if "trycloudflare.com" in line and "https://" in line:
        for w in line.split():
            if w.startswith("https://") and "trycloudflare.com" in w:
                public_url = w.strip()
                break
        if public_url: break

# Drain remaining tunnel output silently
def _drain():
    for _ in tunnel.stdout: pass
threading.Thread(target=_drain, daemon=True).start()

if not public_url:
    raise RuntimeError("Could not extract tunnel URL")

print("\n" + "=" * 60)
print(f"\u2705 YOUR API URL: {public_url}")
print("=" * 60)
print("\nPaste this into .env as COLAB_API_URL and restart your backend.")
